<a href="https://colab.research.google.com/github/ksshah/seed-and-scale/blob/main/summary/components/WiD_Summary_TheBackupGap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Backup Gap - who has no alternative rice supply, and who would have to scale up

*Women in Data "What's Cooking?" Datathon, Sept 2026. Built by Kaveesha, on the pipeline in
`summary/WiD_Summary_TheWaterBill.ipynb` (Heidi) and the rice trade work in
`discovery/QCL_all_Rice_Data_code28.ipynb` (Shruti & Heidi). See `PROJECT_WORKFLOW.md` for how every
notebook in this repo relates.*

**Why this notebook exists.** The Water Bill's shock scenario models a stressed supplier cutting
production, then offers `simulate_shock(..., diversify=True)` as the counterfactual. That path
assumes **unconstrained headroom** at the importer's other suppliers - so the gap always closes to
~0%, for every country, at every shock size. That is the model doing what it was told, not a
finding. It also flatters the answer: for Afghanistan, the suppliers being asked to absorb the loss
are 1.45% of its import volume between them.

This notebook replaces that assumption with a bounded one, and asks the two questions the
assumption was hiding:

1. **Which import-dependent countries have zero real alternative supply today?**
   Scored as an **Alternative Supply Ratio (ASR)** - unstressed import volume divided by stressed
   import volume. An ASR of 0.01 means a country buys one tonne from an unstressed supplier for
   every hundred tonnes it buys from a stressed one.
2. **Which currently-small suppliers would need to scale up to close that gap, and by how much?**
   Answered with a **capacity-constrained waterfall**: rank candidate exporters by water headroom,
   then walk down the list allocating each one's spare capacity (Section 2 defines exactly what
   that is) until the gap is filled or the candidates run out.

Then the question that only shows up once capacity is finite: **what happens if every exposed
country diversifies at the same time?** Spare capacity is a global pool, not a per-country
allowance. Section 6 allocates it once, across all importers, and reports the residual.

**Data sources.** FAOSTAT bulk downloads only, and the same four domains the Water Bill uses:
QCL (production), TCL (trade), the Detailed Trade Matrix (bilateral flows), SDGB (SDG 6.4.2 water
stress). **Food Balance Sheets are deliberately not used here** - FBS "Rice and products" is a
different item aggregation on a milled-equivalent basis, and mixing it with QCL code 27 (paddy) and
TCL codes 28/31 (lifted to paddy-equivalent) produces figures that don't reconcile against the rest
of the pipeline. Everything below stays on the paddy-equivalent basis the team already validated in
`docs/WiD_CodesForRiceAndConfusion.docx`.

**Before this ships**: needs a `Run All` in Colab with outputs saved. It was authored in an
environment where FAOSTAT's bulk-download domain is blocked by network policy, so it has not been
run end to end - do that before the final push, per `GIT_RUNBOOK.md`. Section 1 ends with a
reconciliation cell that checks this notebook's rebuilt pipeline against the Water Bill's published
figures; read that output first.

## Section 0: Setup

In [ ]:
import urllib.request, zipfile, io, os, json, textwrap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Palette shared across every chart below, so the deck reads as one system. Checked for
# colour-vision separation and for contrast against a white slide before being used here.
C_TEAL, C_OCHRE, C_INDIGO, C_SIGNAL = '#00958C', '#CC7A18', '#4E70C4', '#A8442A'
C_GREY = '#8A8F98'

In [ ]:
# Section 0: config. Everything above the divider is carried over VERBATIM from
# WiD_Summary_TheWaterBill.ipynb so the rebuilt pipeline reconciles against the published figures.
# Everything below the divider is new to this notebook and is called out separately, because these
# are the knobs that decide what "spare capacity" means.
CONFIG = {
    # ---- inherited from WiD_Summary_TheWaterBill.ipynb (do not diverge) ----
    'crop_item_code': 27,                 # Rice, paddy (QCL)
    'crop_name': 'Rice',
    'trade_crop_item_code_rm': 31,        # milled/white rice -> paddy-equivalent
    'rm_paddy_factor': 0.67,
    'trade_crop_item_code_rmh': 28,       # husked/brown rice -> paddy-equivalent
    'rmh_paddy_factor': 0.77,
    'aggregate_code_cutoff': 5000,
    'exclude_area_codes': [351, 357, 214, 96, 128],   # China variants -- keep mainland (41) only
    'reexport_threshold_pct': 130,
    'shortlist_min_export_t': 1_000_000,
    'shortlist_min_export_ratio_pct': 5,
    'water_stress_bins': [0, 10, 25, 50, 75, np.inf],
    'water_stress_labels': ['No stress', 'Low stress', 'Medium stress', 'High stress', 'Critical'],
    'stress_threshold_pct': 25,
    'year_min': 2000,
    'year_max': 2024,

    # ---- new in this notebook ----
    # Capacity window. 'recent' is the baseline an exporter is shipping now -- the only window
    # total_headroom_t is measured against.
    'recent_window_years': 3,

    # Total spare capacity (Section 2): total_headroom_t = (recent_production_t - recent_export_t)
    # * divertible_share_of_retained -- a configured share of the production a supplier currently
    # keeps at home. This is an ASSUMPTION, not a measurement -- it presumes a country would
    # redirect part of its domestic crop to export, which has a political and food-security cost
    # this notebook does not model. Set to 0.0 and every supplier's headroom goes to zero.
    'divertible_share_of_retained': 0.05,

    # Which water-stress tiers qualify as real backup supply, for the scale-up recommendation pool
    # (Sections 2, 5, 6) -- a list, checked with .isin(), so it can hold more than one tier. Team
    # call: it's OK to route more export volume to a country at 10-25% withdrawal (Low stress), not
    # just under 10% (No stress). Set back to ['No stress'] for the stricter measured-only answer.
    'backup_requires_tier': ['No stress', 'Low stress'],

    # Minimum spare capacity for a supplier to count as a real scale-up candidate, not a rounding
    # artifact. Countries with essentially zero rice trade (e.g. Iceland) can otherwise show a
    # trivial "headroom" from a one-off historical export blip -- that's noise, not real capacity,
    # but it still ranks well on water stress and pollutes the ranking with a supplier that can't
    # meaningfully help. 10,000 t is a starting point, not a validated number.
    'candidate_min_headroom_t': 10_000,

    # Alternative Supply Ratio classification. ASR = unstressed volume / stressed volume.
    'asr_bins': [0, 0.05, 0.25, 1.0, np.inf],
    'asr_labels': ['No real alternative', 'Thin', 'Partial', 'Adequate'],

    # Shock size used for the scale-up sections. Same 10/20/30 ladder as the Water Bill; 20% is the
    # one the charts use.
    'shock_pct_ladder': [0.10, 0.20, 0.30],
    'shock_pct_headline': 0.20,
}

print('Divertible share of retained production:', CONFIG['divertible_share_of_retained'],
      '  <- this alone now defines total_headroom_t (Section 2); 0.0 means zero headroom')

## Section 1: Rebuild the Water Bill pipeline base
(identical logic to `WiD_Summary_TheWaterBill.ipynb` Sections 1-3 and its water-stress join, so the
two notebooks reconcile; credit: Heidi, Shruti)

Nothing new is computed in this section. It exists so this notebook stands alone in Colab and so the
reconciliation cell at the end of the section can prove the rebuild matches the published figures
before anything is built on top of it.

In [ ]:
# Section 1a: QCL bulk download (Production: Crops and livestock products)
BULK_URL = 'https://bulks-faostat.fao.org/production/Production_Crops_Livestock_E_All_Data_(Normalized).zip'
if not os.path.exists('qcl_bulk.zip'):
    print('Downloading QCL bulk file...')
    urllib.request.urlretrieve(BULK_URL, 'qcl_bulk.zip')

with zipfile.ZipFile('qcl_bulk.zip') as z:
    csv_name = [n for n in z.namelist() if n.endswith('.csv') and 'Flags' not in n
                and 'ItemCodes' not in n and 'AreaCodes' not in n][0]
    qcl_all = pd.read_csv(z.open(csv_name), encoding='latin-1', low_memory=False)
    area_csv = [n for n in z.namelist() if 'AreaCodes' in n][0]
    area_codes = pd.read_csv(z.open(area_csv), encoding='latin-1')

print(f'Full QCL: {len(qcl_all):,} rows')

In [ ]:
# Section 1b: rice production (paddy, tonnes), aggregates and China double-counts dropped.
qcl_rice = qcl_all[(qcl_all['Item Code'] == CONFIG['crop_item_code'])
                   & (qcl_all['Element'] == 'Production')].copy()

area_code_keep = area_codes[
    (area_codes['Area Code'] < CONFIG['aggregate_code_cutoff'])
    & (~area_codes['Area Code'].isin(CONFIG['exclude_area_codes']))
]
valid_country_codes = set(area_code_keep['Area Code'].astype(str))
qcl_rice = qcl_rice[qcl_rice['Area Code'].astype(str).isin(valid_country_codes)].copy()

# Annual production series -- this notebook needs the per-year detail, not just the 2000-2024 total,
# because capacity is measured against a country's own recent years.
rice_by_area_year = (qcl_rice.groupby(['Area', 'Area Code', 'Year'])['Value']
                     .sum().reset_index(name='production_t'))

rice_production_2000_2024 = (
    rice_by_area_year[rice_by_area_year['Year'].between(CONFIG['year_min'], CONFIG['year_max'])]
    .groupby('Area')['production_t'].sum().reset_index()
)
print(f"{rice_by_area_year['Area'].nunique()} producing countries, "
      f"{rice_by_area_year['Year'].min()}-{rice_by_area_year['Year'].max()}")

In [ ]:
# Section 1c: TCL bulk download (Trade: Crops and livestock products)
TCL_URL = 'https://bulks-faostat.fao.org/production/Trade_CropsLivestock_E_All_Data_(Normalized).zip'
if not os.path.exists('tcl_bulk.zip'):
    print('Downloading TCL bulk file...')
    urllib.request.urlretrieve(TCL_URL, 'tcl_bulk.zip')

with zipfile.ZipFile('tcl_bulk.zip') as z:
    tcl_csv_name = [n for n in z.namelist() if n.endswith('.csv') and 'Flags' not in n
                    and 'ItemCodes' not in n and 'AreaCodes' not in n][0]
    tcl_all = pd.read_csv(z.open(tcl_csv_name), encoding='latin-1', low_memory=False)
    tcl_area_csv = [n for n in z.namelist() if 'AreaCodes' in n][0]
    tcl_area_codes = pd.read_csv(z.open(tcl_area_csv), encoding='latin-1')

print(f'Full TCL: {len(tcl_all):,} rows')

In [ ]:
# Section 1d: exports for the two rice trade items, each lifted to paddy-equivalent with its OWN
# factor before summing (28/0.77 + 31/0.67 -- see docs/WiD_CodesForRiceAndConfusion.docx).
rm_code,  rm_factor  = CONFIG['trade_crop_item_code_rm'],  CONFIG['rm_paddy_factor']
rmh_code, rmh_factor = CONFIG['trade_crop_item_code_rmh'], CONFIG['rmh_paddy_factor']

tcl_rice = tcl_all[(tcl_all['Item Code'].isin([rm_code, rmh_code]))
                   & (tcl_all['Element'] == 'Export quantity')].copy()
tcl_rice['paddy_factor'] = tcl_rice['Item Code'].map({rm_code: rm_factor, rmh_code: rmh_factor})
tcl_rice['exported_paddy_eq_t'] = tcl_rice['Value'] / tcl_rice['paddy_factor']

tcl_area_keep = tcl_area_codes[
    (tcl_area_codes['Area Code'] < CONFIG['aggregate_code_cutoff'])
    & (~tcl_area_codes['Area Code'].isin(CONFIG['exclude_area_codes']))
]
valid_tcl_codes = set(tcl_area_keep['Area Code'].astype(str))
tcl_rice = tcl_rice[tcl_rice['Area Code'].astype(str).isin(valid_tcl_codes)].copy()

rice_exports_by_year = (tcl_rice.groupby(['Area', 'Area Code', 'Year'])['exported_paddy_eq_t']
                        .sum().reset_index(name='exported_t'))

rice_exports_eq_2000_2024 = (
    rice_exports_by_year[rice_exports_by_year['Year'].between(CONFIG['year_min'], CONFIG['year_max'])]
    .groupby('Area')['exported_t'].sum().reset_index()
)
print(f"{rice_exports_by_year['Area'].nunique()} exporting countries, "
      f"{rice_exports_by_year['Year'].min()}-{rice_exports_by_year['Year'].max()}")

In [ ]:
# Section 1e: producer/re-exporter classification and the dynamic REPORTERS shortlist
# (same rules as the Water Bill, so the Trade Matrix pull below covers the same countries).
rice_production_export = rice_production_2000_2024.merge(
    rice_exports_eq_2000_2024, on='Area', how='outer')
rice_production_export[['production_t', 'exported_t']] = (
    rice_production_export[['production_t', 'exported_t']].fillna(0))

producers = rice_production_export[rice_production_export['production_t'] > 0].copy()
producers['export_ratio_of_production'] = (
    producers['exported_t'] / producers['production_t'] * 100)
producers = producers[producers['export_ratio_of_production'] <= CONFIG['reexport_threshold_pct']]

shortlist = producers[
    (producers['exported_t'] > CONFIG['shortlist_min_export_t'])
    & (producers['export_ratio_of_production'] > CONFIG['shortlist_min_export_ratio_pct'])
].sort_values('exported_t', ascending=False)

REPORTERS = shortlist['Area'].tolist()
print(f'{len(REPORTERS)} reporters selected for the Trade Matrix pull')
print(shortlist[['Area', 'exported_t', 'export_ratio_of_production']].head(15).to_string(index=False))

In [ ]:
# Section 1f: Detailed Trade Matrix -- large file, several minutes. Filtered inline to the
# shortlisted reporters, the two rice trade item codes, Export quantity, 2000-2024.
TM_BULK = 'https://bulks-faostat.fao.org/production/Trade_DetailedTradeMatrix_E_All_Data_(Normalized).zip'
if not os.path.exists('tm_bulk.zip'):
    print('Downloading TM bulk (~400 MB zip / ~1.5 GB unzipped)...')
    urllib.request.urlretrieve(TM_BULK, 'tm_bulk.zip')

ITEM_CODES = [rm_code, rmh_code]
ELEMENTS = [5910]   # Export quantity (reporter's exports TO each partner)

with zipfile.ZipFile('tm_bulk.zip') as z:
    def is_meta(n):
        low = n.lower().replace('_', '')
        return any(k in low for k in ('flags', 'itemcodes', 'elements', 'countrycodes', 'areacodes'))
    tm_csv_name = [n for n in z.namelist() if n.endswith('.csv') and not is_meta(n)][0]
    print(f'Streaming {tm_csv_name}...')

    parts = []
    for chunk in pd.read_csv(z.open(tm_csv_name), encoding='latin-1',
                             low_memory=False, chunksize=1_000_000):
        chunk.columns = [c.strip() for c in chunk.columns]
        chunk['Item Code'] = pd.to_numeric(chunk['Item Code'], errors='coerce')
        chunk['Element Code'] = pd.to_numeric(chunk['Element Code'], errors='coerce')
        chunk['Year'] = pd.to_numeric(chunk['Year'], errors='coerce')
        m = (chunk['Reporter Countries'].isin(REPORTERS)
             & chunk['Item Code'].isin(ITEM_CODES)
             & chunk['Element Code'].isin(ELEMENTS)
             & chunk['Year'].between(CONFIG['year_min'], CONFIG['year_max']))
        if m.any():
            parts.append(chunk.loc[m])

if not parts:
    raise ValueError('No rows matched -- check reporter spelling / item code coverage.')
tm_rice = pd.concat(parts, ignore_index=True)
print(f"rows: {len(tm_rice):,}   reporters found: {tm_rice['Reporter Countries'].nunique()}/{len(REPORTERS)}")

In [ ]:
# Section 1g: bilateral flows and each importer's dependency on each supplier. Same denominator
# caveat as the Water Bill: shares are of what the importer buys FROM THE REPORTER SET, which is
# the only bilateral data this pipeline pulls -- not of its imports from the whole world.
flows = (tm_rice.groupby(['Reporter Countries', 'Partner Countries'])['Value']
         .sum().reset_index(name='export_qty_t'))
flows = flows[flows['export_qty_t'] > 0].copy()

partner_totals = (flows.groupby('Partner Countries')['export_qty_t']
                  .sum().reset_index(name='partner_total_from_reporters'))
dependency = flows.merge(partner_totals, on='Partner Countries')
dependency['dependency_pct'] = (
    dependency['export_qty_t'] / dependency['partner_total_from_reporters'] * 100).round(1)

print(f"{len(dependency):,} flows, {dependency['Partner Countries'].nunique()} importers, "
      f"{dependency['Reporter Countries'].nunique()} exporters")

In [ ]:
# Section 1h: SDGB water stress (SDG 6.4.2), latest reported year per country, never imputed.
SDG_BULK = 'https://bulks-faostat.fao.org/production/SDG_BulkDownloads_E_All_Data_(Normalized).zip'
if not os.path.exists('sdgb_bulk.zip'):
    print('Downloading SDGB bulk file...')
    urllib.request.urlretrieve(SDG_BULK, 'sdgb_bulk.zip')

with zipfile.ZipFile('sdgb_bulk.zip') as z:
    sdgb_csv_name = [n for n in z.namelist() if n.endswith('.csv') and 'Flags' not in n
                     and 'ItemCodes' not in n and 'AreaCodes' not in n][0]
    sdgb_all = pd.read_csv(z.open(sdgb_csv_name), encoding='latin-1', low_memory=False)

water_stress_all = sdgb_all[sdgb_all['Item Code'] == '24027-_T'].copy()
water_stress_all = water_stress_all.dropna(subset=['Value']).copy()
water_stress_all['Value'] = pd.to_numeric(water_stress_all['Value'], errors='coerce')
water_stress_all = water_stress_all.dropna(subset=['Value']).copy()

idx_latest = water_stress_all.groupby('Area Code')['Year'].idxmax()
water_stress_latest = water_stress_all.loc[
    idx_latest, ['Area', 'Area Code', 'Year', 'Value']].copy()
water_stress_latest = water_stress_latest.rename(
    columns={'Year': 'ws_year', 'Value': 'water_stress_pct'})

water_stress_latest['water_stress_tier'] = pd.cut(
    water_stress_latest['water_stress_pct'], bins=CONFIG['water_stress_bins'],
    labels=CONFIG['water_stress_labels'], right=False)
water_stress_latest['is_stressed'] = (
    water_stress_latest['water_stress_pct'] >= CONFIG['stress_threshold_pct'])

print(f'Countries with a water-stress reading: {len(water_stress_latest)}')
print(water_stress_latest['water_stress_tier'].value_counts().sort_index().to_string())

In [ ]:
# Section 1i: attach supplier-side water stress to every flow. Unmatched names are kept and
# labelled 'No data' rather than dropped, so "not yet known" never silently reads as "not stressed".
NAME_ALIASES = {
    # left: name as it appears in TM/QCL/TCL -> right: name as it appears in SDGB, if different.
    # Populate from the mismatch report printed below; empty until proven necessary.
}

ws_lookup = water_stress_latest[
    ['Area', 'ws_year', 'water_stress_pct', 'water_stress_tier', 'is_stressed']
].rename(columns={'Area': 'Reporter Countries'})

dependency_ws = dependency.copy()
dependency_ws['Reporter Countries'] = dependency_ws['Reporter Countries'].map(
    lambda n: NAME_ALIASES.get(n, n))
dependency_ws = dependency_ws.merge(ws_lookup, on='Reporter Countries', how='left')
dependency_ws['water_stress_tier'] = (
    dependency_ws['water_stress_tier'].astype(object).fillna('No data'))
dependency_ws['is_stressed'] = dependency_ws['is_stressed'].fillna(False)

unmatched = sorted(set(dependency_ws.loc[
    dependency_ws['water_stress_tier'] == 'No data', 'Reporter Countries']))
print(f'Suppliers matched to a water-stress reading: '
      f"{dependency_ws['Reporter Countries'].nunique() - len(unmatched)}"
      f"/{dependency_ws['Reporter Countries'].nunique()}")
if unmatched:
    print('!! NOT matched (add to NAME_ALIASES once you confirm the SDGB spelling):')
    print(unmatched)

In [ ]:
# Section 1j: RECONCILIATION against WiD_Summary_TheWaterBill.ipynb's published figures.
# Read this before trusting anything below it. Flags rather than raises, matching the team's
# credibility-check pattern -- a mismatch is a conversation, not a crash.
PUBLISHED = [
    ('Afghanistan dependency on Pakistan (%)', 98.5,
     lambda: float(dependency_ws[(dependency_ws['Partner Countries'] == 'Afghanistan')
                                 & (dependency_ws['Reporter Countries'] == 'Pakistan')
                                 ]['dependency_pct'].iloc[0])),
    ('Kazakhstan dependency on Pakistan (%)', 82.6,
     lambda: float(dependency_ws[(dependency_ws['Partner Countries'] == 'Kazakhstan')
                                 & (dependency_ws['Reporter Countries'] == 'Pakistan')
                                 ]['dependency_pct'].iloc[0])),
    ('Kenya dependency on Pakistan (%)', 66.3,
     lambda: float(dependency_ws[(dependency_ws['Partner Countries'] == 'Kenya')
                                 & (dependency_ws['Reporter Countries'] == 'Pakistan')
                                 ]['dependency_pct'].iloc[0])),
    ('Pakistan water stress (%)', 98.12,
     lambda: float(water_stress_latest[water_stress_latest['Area'] == 'Pakistan'
                                       ]['water_stress_pct'].iloc[0])),
    ('Egypt water stress (%)', 112.93,
     lambda: float(water_stress_latest[water_stress_latest['Area'] == 'Egypt'
                                       ]['water_stress_pct'].iloc[0])),
    ('Afghanistan total from reporters (t)', 4738866.86,
     lambda: float(dependency_ws[dependency_ws['Partner Countries'] == 'Afghanistan'
                                 ]['export_qty_t'].sum())),
]

recon_rows = []
for label, published, fn in PUBLISHED:
    try:
        computed = fn()
        drift = abs(computed - published) / published * 100 if published else np.nan
        status = 'MATCH' if drift < 1 else f'DRIFT {drift:.1f}% -- investigate before using'
    except (IndexError, KeyError):
        computed, status = np.nan, 'NOT FOUND -- name or pipeline changed'
    recon_rows.append({'check': label, 'water_bill': published,
                       'this_notebook': computed, 'status': status})

reconciliation = pd.DataFrame(recon_rows)
print(reconciliation.to_string(index=False))
print('\nAny row that is not MATCH means this notebook and the Water Bill are computing different '
      'things -- resolve that before reading Sections 2-5.')

## [TODO-8 | Backup Capacity] Section 2: how much spare export capacity each supplier actually has
(replaces the unconstrained-headroom assumption in `WiD_Summary_TheWaterBill.ipynb` §TODO-5,
Heidi; production/export series from `discovery/QCL_all_Rice_Data_code28.ipynb`, Shruti & Heidi)

**Headroom definition:**

```
total_headroom_t = (recent_production_t - recent_export_t) * CONFIG['divertible_share_of_retained']
```

i.e. a configured share of the production a supplier currently keeps at home. This is an
**assumption, not a measurement** - it presumes a country would redirect part of its domestic crop
to export, a political and food-security cost this notebook does not model. Set
`divertible_share_of_retained` to 0.0 and every supplier's headroom is zero.

An earlier version of this notebook also tracked each supplier's best export year in the last 10
(`demonstrated_spare_t`, `peak_export_t`, `peak_year`) and added that measured figure into
`total_headroom_t`, alongside the divertible-share assumption. That machinery has been removed
entirely -- headroom is now defined by the one-line formula above and nothing else, so the output
below only ever carries fields that actually feed the number.

In [ ]:
# Per-exporter capacity headroom, on the paddy-equivalent basis used throughout.
latest_year = int(rice_exports_by_year['Year'].max())
recent_years = list(range(latest_year - CONFIG['recent_window_years'] + 1, latest_year + 1))
print(f'Recent window: {recent_years}')

recent_exp = (rice_exports_by_year[rice_exports_by_year['Year'].isin(recent_years)]
              .groupby('Area')['exported_t'].mean().rename('recent_export_t'))
recent_prod = (rice_by_area_year[rice_by_area_year['Year'].isin(recent_years)]
               .groupby('Area')['production_t'].mean().rename('recent_production_t'))

supplier_capacity = pd.concat([recent_exp, recent_prod], axis=1).reset_index()
supplier_capacity = supplier_capacity.rename(columns={'index': 'Area'})
supplier_capacity[['recent_export_t', 'recent_production_t']] = (
    supplier_capacity[['recent_export_t', 'recent_production_t']].fillna(0))

# Headroom = divertible only: (recent_production_t - recent_export_t) * divertible_share_of_retained.
supplier_capacity['total_headroom_t'] = (
    (supplier_capacity['recent_production_t'] - supplier_capacity['recent_export_t']).clip(lower=0)
    * CONFIG['divertible_share_of_retained'])
supplier_capacity['export_share_of_production_pct'] = np.where(
    supplier_capacity['recent_production_t'] > 0,
    supplier_capacity['recent_export_t'] / supplier_capacity['recent_production_t'] * 100, np.nan)

supplier_capacity = supplier_capacity.merge(
    water_stress_latest[['Area', 'water_stress_pct', 'water_stress_tier', 'is_stressed']],
    on='Area', how='left')
supplier_capacity['water_stress_tier'] = (
    supplier_capacity['water_stress_tier'].astype(object).fillna('No data'))
supplier_capacity['is_stressed'] = supplier_capacity['is_stressed'].fillna(False)

print(supplier_capacity.sort_values('total_headroom_t', ascending=False)
      .head(20)[['Area', 'recent_production_t', 'recent_export_t', 'total_headroom_t',
                 'water_stress_tier']].to_string(index=False))

In [ ]:
# The candidate pool: suppliers that could take on volume WITHOUT moving the problem somewhere else.
# Qualifying tiers are set in CONFIG['backup_requires_tier'] (a list, checked with .isin()). Headroom
# must also clear candidate_min_headroom_t -- a few tonnes of "spare capacity" from a country with no
# real rice trade (e.g. Iceland) is a rounding artifact, not a usable backup supplier.
candidates = supplier_capacity[
    (supplier_capacity['water_stress_tier'].isin(CONFIG['backup_requires_tier']))
    & (supplier_capacity['total_headroom_t'] > CONFIG['candidate_min_headroom_t'])
].sort_values('total_headroom_t', ascending=False).reset_index(drop=True)

print(f"{len(candidates)} suppliers with headroom "
      f"({'/'.join(CONFIG['backup_requires_tier'])} tier, "
      f">{CONFIG['candidate_min_headroom_t']:,} t headroom)")
print(candidates[['Area', 'recent_export_t', 'total_headroom_t', 'water_stress_pct']]
      .to_string(index=False))
print(f"\nTOTAL candidate headroom: {candidates['total_headroom_t'].sum():,.0f} t "
      f"(all assumed -- divertible_share_of_retained, see Section 2)")

In [ ]:
# Where the world's rice capacity sits relative to its water stress. The bottom-right quadrant is
# the one that matters: big headroom, low stress. If it is thin, resilience advice has nowhere to go.
plot_df = supplier_capacity[(supplier_capacity['total_headroom_t'] > 0)
                            & (supplier_capacity['water_stress_pct'].notna())].copy()

fig, ax = plt.subplots(figsize=(13, 7.5))
colors = np.where(plot_df['is_stressed'], C_SIGNAL, C_TEAL)
ax.scatter(plot_df['total_headroom_t'] / 1e6, plot_df['water_stress_pct'],
           s=60, c=colors, alpha=0.85, edgecolors='white', linewidths=1.2, zorder=3)

# Headroom spans several orders of magnitude (a few huge exporters vs. many small ones) -- on a
# linear axis that pins almost every country into a sliver near zero. Log-x spreads them back out.
ax.set_xscale('log')

ax.axhline(CONFIG['stress_threshold_pct'], linestyle='--', color=C_GREY, linewidth=1, zorder=1)
ax.text(0.99, CONFIG['stress_threshold_pct'] + 2,
        f"water-stress threshold ({CONFIG['stress_threshold_pct']}%)",
        transform=ax.get_yaxis_transform(), ha='right', fontsize=9, color=C_GREY)

top_labelled = plot_df.sort_values('total_headroom_t', ascending=False).head(15)
for _, r in top_labelled.iterrows():
    ax.annotate(r['Area'], (r['total_headroom_t'] / 1e6, r['water_stress_pct']),
                xytext=(6, 4), textcoords='offset points', fontsize=9)

ax.set_xlabel('Spare export capacity (million tonnes, paddy-equivalent, log scale)')
ax.set_ylabel('Freshwater withdrawal, SDG 6.4.2 (%)')
ax.set_title('Rice exporters: spare capacity vs. their own water stress', fontsize=13)
ax.set_yscale('symlog', linthresh=100)
ax.grid(True, which='both', alpha=0.25, zorder=0)

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C_TEAL, markersize=9,
           label='Unstressed - usable as backup'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C_SIGNAL, markersize=9,
           label='Already water-stressed - moves the problem'),
], loc='upper left', fontsize=10, frameon=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/backup_capacity_vs_stress.png', dpi=150)
plt.show()

### Building blocks used by the spotlight below

`alternatives` (ASR per importer) and the `scale_up_plan` machinery are computed here, ahead of their own sections, purely because the Afghanistan spotlight in Section 3 needs both before the full-world ranking (Section 4) and scale-up tables (Section 5) are revealed. Nothing here changes what they compute -- Sections 4 and 5 build on these exact same objects.

`scale_up_plan` returns two independent plans, not one: a **primary** plan (the nearest-ranked candidates) and a **secondary/backup** plan (`backup_*` fields) built from whoever is left in the same ranked pool after the primary suppliers are set aside -- i.e. "if every primary supplier fell through, who else could still close this same gap." The secondary plan is a full Plan B sized to the same gap, not extra margin stacked on top of the primary one.

**Units: `alternatives`' tonnage columns (and everything derived from them, including `gap_t`) are annual averages, not the 25-year cumulative totals in `dependency_ws`.** `dependency_ws`/`flows` (Section 1g) sum FAOSTAT's Detailed Trade Matrix over the full 2000-2024 pull -- fine for a ratio like `dependency_pct`, where the 25-year window cancels out, but not for an absolute tonnage compared against `total_headroom_t` (Section 2), which is already a recent-year rate. `summarize_alternatives` below divides by the number of years in the pull (25) before anything downstream uses it, matching the same "25-year total / 25" annualization `water-bill.html` already applies to its own figures -- so `gap_t` and `total_headroom_t` are both t/year and are safe to compare directly in `close_gap`.

In [ ]:
def rank_candidates(importer, candidates, dependency_ws):
    """Candidate suppliers for one importer, ranked by whether a trade relationship already
    exists first, then by water stress (least stressed first), then by size. `candidates` is
    already filtered to CONFIG['backup_requires_tier'] (No stress/Low stress) before it ever
    reaches this function, so every row here already clears that water-stress bar -- this
    function only decides ORDER among suppliers that already qualify, it does not re-filter."""
    existing = set(dependency_ws.loc[dependency_ws['Partner Countries'] == importer,
                                     'Reporter Countries'])
    c = candidates.copy()
    c['already_supplies'] = c['Area'].isin(existing)
    return c.sort_values(['already_supplies', 'water_stress_pct', 'total_headroom_t'],
                         ascending=[False, True, False]).reset_index(drop=True)


def close_gap(gap_t, ranked_candidates, capacity_col='total_headroom_t'):
    """Walk down the ranked candidates allocating capacity until the gap is filled or they run out.
    Returns (filled_t, allocations) where allocations is a list of (supplier, tonnes, is_new).
    Every row it visits before stopping ends up in `allocations` (capacity is always > 0 for a
    candidate, by construction of the candidate pool), so `allocations` always corresponds to an
    exact PREFIX of `ranked_candidates` -- callers rely on this to slice off a used prefix and
    build a backup layer from what's left (see scale_up_plan's `backup_*` fields)."""
    filled, allocations = 0.0, []
    for _, row in ranked_candidates.iterrows():
        remaining = gap_t - filled
        if remaining <= 1e-6:
            break
        take = min(remaining, float(row[capacity_col]))
        if take > 0:
            filled += take
            allocations.append((row['Area'], take, not bool(row['already_supplies'])))
    return filled, allocations


def scale_up_plan(importer, shock_pct, candidates, dependency_ws, alternatives):
    """What it would take to replace `shock_pct` of an importer's stressed supply -- as a PRIMARY
    plan (the nearest-ranked suppliers), and as a SECONDARY/backup plan (what's left in the same
    ranked pool if every primary supplier turned out to be unavailable). The secondary plan is
    built by continuing down the exact same ranked list from where the primary allocation left
    off, and sizing it to close the SAME gap again -- it is not extra margin on top of the primary
    plan, it is an independent Plan B that assumes Plan A fell through entirely."""
    row = alternatives[alternatives['importer'] == importer]
    if row.empty:
        return None
    gap = float(row['stressed_t'].iloc[0]) * shock_pct
    ranked = rank_candidates(importer, candidates, dependency_ws)

    filled, allocations = close_gap(gap, ranked)
    backup_pool = ranked.iloc[len(allocations):].reset_index(drop=True)
    backup_filled, backup_allocations = close_gap(gap, backup_pool)

    return {
        'importer': importer,
        'shock_pct': int(shock_pct * 100),
        'gap_t': gap,
        'filled_t': filled,
        'closed_pct': filled / gap * 100 if gap > 0 else np.nan,
        'residual_t': max(0.0, gap - filled),
        'n_suppliers_needed': len(allocations),
        'n_new_relationships': sum(1 for _, _, is_new in allocations if is_new),
        'allocations': allocations,
        'backup_filled_t': backup_filled,
        'backup_closed_pct': backup_filled / gap * 100 if gap > 0 else np.nan,
        'backup_residual_t': max(0.0, gap - backup_filled),
        'n_backup_suppliers_needed': len(backup_allocations),
        'n_backup_new_relationships': sum(1 for _, _, is_new in backup_allocations if is_new),
        'backup_allocations': backup_allocations,
    }

In [ ]:
# dependency_ws's export_qty_t is the 2000-2024 CUMULATIVE sum from Section 1g. Divide by the
# number of years in that pull to get an annual average -- the same "25-year total / 25"
# annualization water-bill.html applies -- before it becomes an absolute tonnage anywhere
# downstream (gap_t in scale_up_plan, MIN_VOLUME_T in Section 4). Percentages and the ASR are
# ratios of these same annualized numbers, so they come out numerically identical either way.
N_YEARS = CONFIG['year_max'] - CONFIG['year_min'] + 1

def summarize_alternatives(g):
    total = g['export_qty_t'].sum() / N_YEARS
    stressed = g.loc[g['is_stressed'], 'export_qty_t'].sum() / N_YEARS
    no_stress = g.loc[g['water_stress_tier'] == 'No stress', 'export_qty_t'].sum() / N_YEARS
    low_stress = g.loc[g['water_stress_tier'] == 'Low stress', 'export_qty_t'].sum() / N_YEARS
    nodata = g.loc[g['water_stress_tier'] == 'No data', 'export_qty_t'].sum() / N_YEARS
    # "Unstressed" for the ASR = anything BELOW the project's own water-stress threshold
    # (stress_threshold_pct, 25%) -- i.e. No stress + Low stress, the complement of is_stressed.
    # This is fixed to the 25% threshold on purpose, independent of CONFIG['backup_requires_tier']
    # (used in Section 2/5/6 for the scale-up candidate pool): this section measures an importer's
    # CURRENT exposure, which shouldn't silently change if that separate recommendation knob is
    # ever narrowed back to just No stress.
    unstressed = no_stress + low_stress
    return pd.Series({
        'total_from_reporters_t': total,
        'n_suppliers': len(g),
        'stressed_t': stressed,
        'unstressed_t': unstressed,
        'no_stress_t': no_stress,
        'low_stress_t': low_stress,
        'stressed_pct': stressed / total * 100 if total else np.nan,
        'unstressed_pct': unstressed / total * 100 if total else np.nan,
        'no_stress_pct': no_stress / total * 100 if total else np.nan,
        'low_stress_pct': low_stress / total * 100 if total else np.nan,
        'nodata_pct': nodata / total * 100 if total else np.nan,
        'asr': (unstressed / stressed) if stressed > 0 else np.inf,
    })

alternatives = (dependency_ws.groupby('Partner Countries')
                .apply(summarize_alternatives, include_groups=False)
                .reset_index().rename(columns={'Partner Countries': 'importer'}))

alternatives['asr_class'] = pd.cut(
    alternatives['asr'], bins=CONFIG['asr_bins'], labels=CONFIG['asr_labels'], right=False)

# pd.cut leaves an INFINITE ratio unclassified (inf < inf is False), but an infinite ASR means the
# importer buys from no water-stressed supplier at all -- the safest case there is, not a blank.
alternatives.loc[np.isinf(alternatives['asr']), 'asr_class'] = CONFIG['asr_labels'][-1]
alternatives['n_suppliers'] = alternatives['n_suppliers'].astype(int)

# An ASR built mostly on suppliers with no SDG 6.4.2 reading is not a finding, it is a data gap
# wearing one. Flag those rather than letting them rank as if they were measured.
alternatives['asr_reliable'] = alternatives['nodata_pct'] < 25
alternatives = alternatives.sort_values('asr').reset_index(drop=True)

In [ ]:
# Shock size used everywhere below: scale-up plans, the pooled allocation, the headline chart.
SHOCK = CONFIG['shock_pct_headline']

## [TODO-8 | Backup Capacity] Section 3: Afghanistan, a worked example

Before looking at the world, look at the country that motivated this whole notebook. In `WiD_Summary_TheWaterBill.ipynb`, Afghanistan buys 98.5% of its rice from Pakistan alone -- a supplier already at 98.12% freshwater withdrawal -- and that notebook's `diversify=True` path claims the gap always closes to ~0%, because it assumes unconstrained headroom anywhere. Here is what actually happens once that assumption is replaced with measured capacity and a real water-stress floor on who counts as a backup supplier.

(This is the same named scale-up plan used for the video/deck. `FEATURED` is hardcoded to `'Afghanistan'` here rather than picked dynamically, so it can't silently point at a different country if the underlying data changes.)

In [ ]:
FEATURED = 'Afghanistan'

row = alternatives[alternatives['importer'] == FEATURED].iloc[0]
plan = scale_up_plan(FEATURED, SHOCK, candidates, dependency_ws, alternatives)

afg_pk_dependency = float(dependency_ws[(dependency_ws['Partner Countries'] == FEATURED)
                                        & (dependency_ws['Reporter Countries'] == 'Pakistan')
                                        ]['dependency_pct'].iloc[0])

print(f'=== {FEATURED} ===')
print(f"  Dependency on Pakistan specifically: {afg_pk_dependency:.1f}%")
print(f"  Suppliers: {int(row['n_suppliers'])}   "
      f"From stressed suppliers: {row['stressed_pct']:.1f}%   "
      f"From unstressed: {row['unstressed_pct']:.2f}%")
print(f"  Alternative Supply Ratio: {row['asr']:.3f}  ({row['asr_class']})")
print(f"\n  At a {plan['shock_pct']}% cut to its stressed suppliers it needs "
      f"{plan['gap_t']:,.0f} t/year replaced.")

print(f"\n  PRIMARY plan: candidates can supply {plan['filled_t']:,.0f} t/year "
      f"({plan['closed_pct']:.1f}% of the gap), "
      f"leaving {plan['residual_t']:,.0f} t/year unmet.")
print(f"  It would take {plan['n_suppliers_needed']} suppliers, of which "
      f"{plan['n_new_relationships']} are trade relationships that do not exist today.")
print('  Ranked by tonnes supplied (descending):')
for supplier, tonnes, is_new in sorted(plan['allocations'], key=lambda a: a[1], reverse=True):
    tag = 'NEW relationship' if is_new else 'existing supplier'
    print(f'    {supplier:<32} {tonnes:>14,.0f} t/year   ({tag})')

print(f"\n  SECONDARY plan (Plan B -- if every primary supplier above were unavailable):")
print(f"  the next-ranked candidates could supply {plan['backup_filled_t']:,.0f} t/year "
      f"({plan['backup_closed_pct']:.1f}% of the same gap), "
      f"leaving {plan['backup_residual_t']:,.0f} t/year unmet.")
print(f"  It would take {plan['n_backup_suppliers_needed']} suppliers, of which "
      f"{plan['n_backup_new_relationships']} are trade relationships that do not exist today.")
print('  Ranked by tonnes supplied (descending):')
for supplier, tonnes, is_new in sorted(plan['backup_allocations'], key=lambda a: a[1], reverse=True):
    tag = 'NEW relationship' if is_new else 'existing supplier'
    print(f'    {supplier:<32} {tonnes:>14,.0f} t/year   ({tag})')

## [TODO-8 | Backup Capacity] Section 4: which importers have zero real alternative supply
(built on `dependency_ws` from Section 1; answers question 1)

**Is Afghanistan a one-off, or does this generalize?** The Alternative Supply Ratio below scores every importer in the dataset the exact same way just used for Afghanistan in Section 3.

**Alternative Supply Ratio (ASR)** = unstressed import volume / stressed import volume, per importer.
`stressed` and `unstressed` split on the *same* UN-Water threshold used everywhere else in this
notebook (`stress_threshold_pct`, 25% freshwater withdrawal): `stressed` = Medium/High/Critical
tier suppliers, `unstressed` = No-stress + Low-stress tier suppliers. `No data` volume is excluded
from both sides, so a country whose suppliers simply haven't reported SDG 6.4.2 is never scored as
though it were safe.

- **ASR 0.00-0.05** - *No real alternative.* For every 100 t bought from a water-stressed supplier,
  under 5 t comes from a supplier below the threshold. There is no relationship to scale up in a
  hurry.
- **0.05-0.25** - *Thin.*  **0.25-1.0** - *Partial.*  **>= 1.0** - *Adequate.*

**This threshold now matches Section 2/5/6's `backup_requires_tier`.** Both this section's ASR and
the scale-up candidate pool treat No-stress + Low-stress suppliers as real, usable supply -- a team
call that it's OK to route export volume to a country at 10-25% withdrawal, not just under 10%.
(Earlier versions of this notebook kept the candidate pool stricter than the ASR; that gap is now
closed. To go back to the stricter, measured-only-safe answer for the candidate pool, set
`CONFIG['backup_requires_tier']` back to `['No stress']`.) `no_stress_pct` and `low_stress_pct` are
still reported separately below so the breakdown within `unstressed_pct` stays visible, and the
distortion this fixed is worth remembering: importers buying ~95% of their rice from a
mildly-stressed supplier (Brunei, Cambodia, Ecuador) used to land in the same "No real alternative"
bucket as importers buying almost entirely from Critical-stress suppliers, purely because neither
cleared the stricter 10% cut.

In [ ]:
print(alternatives['asr_class'].value_counts().reindex(CONFIG['asr_labels']).to_string())
print(f"\nImporters whose ASR rests on >25% 'No data' volume (excluded from the chart below): "
      f"{(~alternatives['asr_reliable']).sum()}")
print(f"\nImporters with NO real alternative supply: "
      f"{(alternatives['asr_class'] == 'No real alternative').sum()} of {len(alternatives)}")

# Sanity check on the redefinition above: an importer buying mostly Low-stress supply should no
# longer land in "No real alternative" just because none of its suppliers clear the strict
# No-stress bar. If this count isn't near zero, a tier-strictness artifact still remains.
no_alt = alternatives[alternatives['asr_class'] == 'No real alternative']
still_mislabeled = no_alt[no_alt['stressed_pct'] < CONFIG['stress_threshold_pct']]
print(f"  of which still look mislabeled by tier strictness (stressed_pct < "
      f"{CONFIG['stress_threshold_pct']}% yet still 'No real alternative'): {len(still_mislabeled)}")
print(f"  of which genuinely buy mostly from water-stressed suppliers (stressed_pct >= "
      f"{CONFIG['stress_threshold_pct']}%): {len(no_alt) - len(still_mislabeled)}")

print('\nLowest 20 by Alternative Supply Ratio:')
print(alternatives.head(20)[['importer', 'total_from_reporters_t', 'n_suppliers', 'stressed_pct',
                             'no_stress_pct', 'low_stress_pct', 'unstressed_pct', 'nodata_pct',
                             'asr', 'asr_class']]
      .to_string(index=False))

In [ ]:
# Only countries buying a meaningful volume -- a tiny importer with one supplier is arithmetically
# extreme but not a food-security story. alternatives' tonnage columns are annual averages (see
# "Building blocks" above), so this is rescaled from the old 500,000 t CUMULATIVE (2000-2024)
# threshold to 500,000 / 25 years, keeping the same effective bar.
MIN_VOLUME_T = 20_000
ranked = alternatives[(alternatives['total_from_reporters_t'] >= MIN_VOLUME_T)
                      & (alternatives['asr_reliable'])].copy()
print(f'{len(ranked)} importers above {MIN_VOLUME_T:,} t/year with a reliable water-stress picture')
top_at_risk = ranked.head(18)

class_color = {'No real alternative': C_SIGNAL, 'Thin': C_OCHRE,
               'Partial': C_INDIGO, 'Adequate': C_TEAL}

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(top_at_risk['importer'][::-1],
               top_at_risk['unstressed_pct'][::-1],
               color=[class_color.get(c, C_GREY) for c in top_at_risk['asr_class'][::-1]])
for bar, val in zip(bars, top_at_risk['unstressed_pct'][::-1]):
    ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}%', va='center', fontsize=8.5, color='#38504e')

ax.set_xlabel(f"Share of rice imports below the water-stress threshold "
              f"({CONFIG['stress_threshold_pct']}%, No stress + Low stress)")
ax.set_title(f'Importers with no real backup supply\n(rice buyers above '
             f'{MIN_VOLUME_T/1e3:.0f}kt/year, annual average 2000-2024)', loc='left')
ax.grid(True, axis='x', alpha=0.25)
ax.set_axisbelow(True)

ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=class_color[l])
                   for l in CONFIG['asr_labels'] if l in set(top_at_risk['asr_class'].dropna())],
          labels=[l for l in CONFIG['asr_labels'] if l in set(top_at_risk['asr_class'].dropna())],
          loc='lower right', fontsize=9, frameon=False, title='Alternative Supply Ratio')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/backup_gap_alternative_supply_ratio.png', dpi=150)
plt.show()

## [TODO-9 | Scale-Up Path] Section 5: who would have to scale up, and by how much
(generalises `alt_suppliers` from `WiD_Summary_TheWaterBill.ipynb` §TODO-5, Heidi; answers question 2)

**Why the Water Bill's `alt_suppliers` ranking couldn't just be reused.** That section ranks an
importer's *existing* suppliers only (`supplier_mix`, built from that importer's own bilateral
flows) by water-stress tier then current import share, and hands the order to
`simulate_shock(diversify=True)`, which reallocates the lost volume across them assuming
**unconstrained headroom** -- exactly the assumption this notebook exists to remove (see the
intro). Two changes were needed versus that approach:

1. **Candidates are not limited to suppliers the importer already buys from.** The Water Bill's
   `supplier_mix` is empty or nearly so for exactly the countries Section 4 flags as having "No
   real alternative" - there is nothing there to rank. Question 2 asks who would *need to* scale
   up, so the pool here is every unstressed exporter with headroom worldwide, existing customer or
   not. Existing relationships are still preferred first in `rank_candidates` below, and the number
   of brand-new relationships a plan would require is reported as its own column - that is a real
   cost, not a free variable.
2. **Each candidate can only give what it has.** The Water Bill's ranking only sets an order; the
   volume math elsewhere assumes every alternative can absorb whatever is asked of it, which is why
   its diversified gap always closes to ~0%, for every country, at every shock size. `close_gap`
   below stops each candidate at its own measured/assumed `total_headroom_t` and reports the
   residual when the pool runs out, instead of always closing the gap.

In [ ]:
# Independent view: every importer is scored as though it were the only country diversifying, which
# is how the Water Bill's diversify=True path implicitly treats them. Includes both the primary plan
# and the secondary/backup plan (see scale_up_plan) -- 'allocations' and 'backup_allocations' are
# list-valued, so they're dropped here the same way; use scale_up_plan directly to inspect either.
independent_rows = []
for importer in ranked['importer']:
    plan = scale_up_plan(importer, SHOCK, candidates, dependency_ws, alternatives)
    if plan:
        independent_rows.append({k: v for k, v in plan.items()
                                  if k not in ('allocations', 'backup_allocations')})

independent = pd.DataFrame(independent_rows).sort_values('gap_t', ascending=False)
print(f'Replacing {int(SHOCK*100)}% of a recent year\'s stressed supply, each importer considered on its own:')
print(independent.head(15)[['importer', 'gap_t', 'closed_pct', 'n_suppliers_needed',
                            'n_new_relationships', 'backup_closed_pct',
                            'n_backup_suppliers_needed', 'n_backup_new_relationships']]
      .to_string(index=False))
print(f"\nFully closable on their own (primary plan): "
      f"{(independent['closed_pct'] >= 99.9).sum()}/{len(independent)} importers")
print(f"Fully closable on their own (secondary/backup plan): "
      f"{(independent['backup_closed_pct'] >= 99.9).sum()}/{len(independent)} importers")

## [TODO-9 | Scale-Up Path] Section 6: what if everyone diversifies at once?
(new to this notebook)

Section 5 gives every importer the whole candidate pool to draw on. In reality the pool is shared:
if a shock hits a major exporter, every one of its buyers reaches for the same alternatives in the
same week. Spare capacity is a **global stock, not a per-country allowance**.

This section allocates that stock **once**, across all importers, most-exposed served first, and
reports what is left over. The gap between Section 5 and Section 6 is the point of the notebook:
advice that works for one country can fail for all of them simultaneously.

*The priority rule - most-exposed first - is a policy choice, not a fact. Change it to largest
volume, or to lowest income, and the winners change. What does not change is the total.*

In [ ]:
# Total demand vs total supply, before any allocation. Both are now annual rates (t/year) --
# gap_t was rescaled from a 25-year cumulative sum in "Building blocks" above specifically so this
# comparison is apples-to-apples with total_headroom_t, which was always a recent-year rate.
total_demand = independent['gap_t'].sum()
total_pool = candidates['total_headroom_t'].sum()
print(f'Demand: {total_demand:,.0f} t/year   Candidate pool: {total_pool:,.0f} t/year')
print(f'The unstressed pool covers {total_pool / total_demand * 100:.1f}% of world demand '
      f'at a {int(SHOCK*100)}% shock.\n')

# Allocate once, most-exposed first. `remaining` is consumed as we go.
priority = (alternatives[alternatives['importer'].isin(independent['importer'])]
            .sort_values('stressed_pct', ascending=False))

remaining = candidates.set_index('Area')['total_headroom_t'].astype(float).copy()
pooled_rows = []
for importer in priority['importer']:
    gap = float(independent.loc[independent['importer'] == importer, 'gap_t'].iloc[0])
    live = candidates[candidates['Area'].isin(remaining[remaining > 0].index)].copy()
    live['total_headroom_t'] = live['Area'].map(remaining)
    ranked_live = rank_candidates(importer, live, dependency_ws)
    filled, allocations = close_gap(gap, ranked_live)
    for supplier, tonnes, _ in allocations:
        remaining[supplier] -= tonnes
    pooled_rows.append({'importer': importer, 'gap_t': gap, 'pooled_filled_t': filled,
                        'pooled_closed_pct': filled / gap * 100 if gap > 0 else np.nan,
                        'n_suppliers_needed': len(allocations)})

pooled = pd.DataFrame(pooled_rows)
comparison = (independent[['importer', 'gap_t', 'closed_pct', 'n_new_relationships']]
              .rename(columns={'closed_pct': 'independent_closed_pct'})
              .merge(pooled[['importer', 'pooled_closed_pct']], on='importer'))

print(f"Fully closable alone      : {(comparison['independent_closed_pct'] >= 99.9).sum()}"
      f"/{len(comparison)}")
print(f"Fully closable once shared: {(comparison['pooled_closed_pct'] >= 99.9).sum()}"
      f"/{len(comparison)}")
print(f"Residual unmet demand     : "
      f"{(comparison['gap_t'] * (1 - comparison['pooled_closed_pct'] / 100)).sum():,.0f} t/year")
print(f"Capacity left in the pool : {remaining[remaining > 0].sum():,.0f} t/year\n")
print(comparison.sort_values('gap_t', ascending=False).head(15).to_string(index=False))

In [ ]:
# The headline chart: the same countries, scored alone vs. sharing one pool.
top_n = comparison.sort_values('gap_t', ascending=False).head(14)
x = np.arange(len(top_n))
width = 0.38

fig, ax = plt.subplots(figsize=(11.5, 5.5))
ax.bar(x - width / 2, top_n['independent_closed_pct'], width,
       label='Diversifying alone', color=C_TEAL)
ax.bar(x + width / 2, top_n['pooled_closed_pct'], width,
       label='Everyone diversifying at once', color=C_SIGNAL)

ax.set_xticks(x)
ax.set_xticklabels(top_n['importer'], rotation=45, ha='right')
ax.set_ylabel('% of the supply gap that can actually be filled')
ax.set_ylim(0, 105)
ax.set_title(f'The same advice, given to one country vs. to all of them\n'
             f'({int(SHOCK*100)}% cut to water-stressed suppliers)', loc='left')
ax.legend(frameon=False, fontsize=9)
ax.grid(True, axis='y', alpha=0.25)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/backup_gap_alone_vs_pooled.png', dpi=150)
plt.show()

## Section 7: exports for the deck

(The featured single-country worked example lives in Section 3 above, computed once there; nothing about it is recomputed here.)

In [ ]:
alternatives.to_csv(f'{OUTPUT_DIR}/backup_gap_alternative_supply_ratio.csv', index=False)
supplier_capacity.to_csv(f'{OUTPUT_DIR}/backup_gap_supplier_capacity.csv', index=False)
candidates.to_csv(f'{OUTPUT_DIR}/backup_gap_candidate_suppliers.csv', index=False)
comparison.to_csv(f'{OUTPUT_DIR}/backup_gap_alone_vs_pooled.csv', index=False)
reconciliation.to_csv(f'{OUTPUT_DIR}/backup_gap_reconciliation.csv', index=False)

print(f'Exported to {OUTPUT_DIR}/:')
for fname in ['backup_gap_alternative_supply_ratio.csv', 'backup_gap_supplier_capacity.csv',
              'backup_gap_candidate_suppliers.csv', 'backup_gap_alone_vs_pooled.csv',
              'backup_gap_reconciliation.csv', 'backup_capacity_vs_stress.png',
              'backup_gap_alternative_supply_ratio.png', 'backup_gap_alone_vs_pooled.png']:
    print(f'  {OUTPUT_DIR}/{fname}')

## Limitations & next steps
*(mirrors the pattern in `WiD_Summary_TheWaterBill.ipynb` and `Barley_Water_Risk_Simulator.ipynb` §9)*

**What's measured:**
- Water stress is each country's latest reported SDG 6.4.2 value, never imputed. Suppliers with no
  reading are labelled `No data` and excluded from *both* sides of the Alternative Supply Ratio, so
  a missing reading can never read as "safe".
- Production and export figures use the conversion logic validated in
  `docs/WiD_CodesForRiceAndConfusion.docx` (QCL 27 for production; TCL 28/0.77 + 31/0.67 for
  exports). Section 1j reconciles the rebuild against the Water Bill's published numbers.

**What's assumed, and where to push back:**
- **All of `total_headroom_t` is the divertible share of retained production:**
  `(recent_production_t - recent_export_t) * divertible_share_of_retained` (default 5%). This is a
  policy assumption, not a measurement - it presumes a country would redirect part of its domestic
  crop to export, which has a food-security cost this notebook does not model. Set it to 0.0 and
  headroom goes to zero for every supplier under this definition.
- **Bilateral shares are of the reporter set, not the world.** Inherited from the Water Bill: the
  Trade Matrix pull only covers shortlisted exporters, so an importer buying from a country outside
  that shortlist has that volume invisible here. This understates alternatives for some importers.
- **Capacity is treated as fungible.** A tonne of Thai spare capacity is assumed deliverable to any
  buyer. Freight, contracts, variety preference and import standards all say otherwise.
- **The priority rule in Section 6** (most-exposed first) determines who gets served, not how much
  exists. Change the rule and individual countries move; the residual does not.
- **No price response.** A shortage raises prices, which draws out supply this model cannot see and
  prices out buyers it does not track. Both effects are real and both are missing.

**Next steps:**
1. Run this end to end in Colab and read Section 1j's reconciliation table before anything else.
2. Sensitivity: re-run with `divertible_share_of_retained` at 0.0 (zero headroom under this
   definition), 0.05, and 0.10, and report the range rather than a point estimate.
3. If the deck needs one number, use Section 6's residual - it holds even when the pool is generous,
   but note it now depends entirely on the divertible-share assumption above, since that is the only
   thing `total_headroom_t` measures.